#### SCHEMA PROFILE

##### 03.1 DOCUMENTAÇÃO DO SCHEMA PROFILE

###### Objetivo

O *Schema Profile* tem como objetivo analisar a estrutura e as características fundamentais do DataFrame.

Enquanto os Profiles seguintes analisam distribuição, padrões, duplicidade, consistência e comportamento estatístico, o *Schema Profile* estabelece uma visão inicial da estrutura dos dados.

A análise considera automaticamente as características das colunas, sem depender do significado de negócio dos atributos.

###### Principais análises

- Estrutura do schema;
- Tipos de dados;
- Quantidade de registros;
- Quantidade de colunas;
- Preenchimento;
- Valores NULL;
- Valores vazios;
- Valores distintos;
- Cardinalidade;
- Completude;
- Classificação estrutural das colunas.

###### Característica

O *Schema Profile* é independente da origem dos dados.

Sua análise é realizada exclusivamente sobre o DataFrame preparado pelo *Data Preparation*.

Dessa forma, o mesmo Profile pode ser utilizado em diferentes fontes e estruturas de dados.

***Cardinalidade***: mede a diversidade de valores em cada coluna, útil para detectar atributos pouco informativos ou com excesso de variação.

###### Resultado esperado

Ao final desta etapa estarão disponíveis:

- métricas estruturais por coluna;
- indicadores de preenchimento;
- indicadores de NULL;
- indicadores de valores vazios;
- quantidade de valores distintos;
- cardinalidade;
- classificação estrutural das colunas;
- DataFrame consolidado do Schema Profile.


In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 03.3 VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame preparado e as informações estruturais necessárias estão disponíveis para execução do Schema Profile.

# COMO FAZ:
# Verifica a existência do DataFrame preparado, das informações do schema e do total de registros calculado anteriormente.

# POR QUE É IMPORTANTE:
# Garante que o Profile seja executado somente após a conclusão correta do Data Preparation, evitando novas ações desnecessárias sobre o DataFrame.

# PERGUNTA RESPONDIDA:
# "O DataFrame preparado possui as informações necessárias para executar o Schema Profile?"

if "df_prepared" not in locals():
    raise ValueError(
        "O DataFrame 'df_prepared' não foi disponibilizado pelo Data Preparation."
    )

if "schema_df" not in locals():
    raise ValueError(
        "O DataFrame 'schema_df' não foi disponibilizado pelo Data Preparation."
    )

if "total_registros" not in locals():
    raise ValueError(
        "A variável 'total_registros' não foi disponibilizada pelo Data Preparation."
    )

if total_registros == 0:
    raise ValueError(
        "O DataFrame preparado não possui registros."
    )

print("DataFrame validado com sucesso.")
print(f"Total de registros: {total_registros}")
print(f"Total de colunas: {len(df_prepared.columns)}")

In [0]:
# ============================================================
# 03.4 INFORMAÇÕES GERAIS
# ============================================================

# O QUE FAZ:
# Organiza as principais informações estruturais do DataFrame que serão utilizadas como referência pelo Schema Profile.

# COMO FAZ:
# Reutiliza as informações já identificadas pelo Data Preparation, evitando nova leitura ou nova contagem do DataFrame.

# POR QUE É IMPORTANTE:
# Centraliza as informações estruturais utilizadas pelo Profile e evita processamento redundante.

# PERGUNTA RESPONDIDA:
# "Qual é a estrutura geral do DataFrame analisado?"

total_colunas = len(df_prepared.columns)

informacoes_gerais = [
    ("Total de registros", total_registros),
    ("Total de colunas", total_colunas),
    ("Colunas numéricas", len(colunas_numericas)),
    ("Colunas de texto", len(colunas_texto)),
    ("Colunas de data", len(colunas_data)),
    ("Colunas de data/hora", len(colunas_data_hora)),
    ("Colunas booleanas", len(colunas_booleanas)),
    ("Outros tipos", len(colunas_outros))
]

informacoes_gerais_df = spark.createDataFrame(
    informacoes_gerais,
    ["metrica", "valor"]
)

display(informacoes_gerais_df)

In [0]:
# ============================================================
# 03.5 CONSTRUÇÃO DAS MÉTRICAS
# ============================================================

# O QUE FAZ:
# Constrói as expressões Spark utilizadas para calcular as métricas estruturais de cada coluna.

# COMO FAZ:
# Cria dinamicamente expressões de agregação para preenchimento, NULL, valores vazios e valores distintos.
# As expressões são construídas antes da execução para permitir que o Spark realize as métricas de forma consolidada.

# POR QUE É IMPORTANTE:
# Evita executar uma ação Spark separada para cada coluna, reduzindo a quantidade de leituras e operações no cluster.

# PERGUNTA RESPONDIDA:
# "Quais métricas estruturais serão calculadas para cada coluna?"

metricas_schema = []

for campo in df_prepared.schema.fields:

    coluna = campo.name
    coluna_ref = F.col(coluna)

    # --------------------------------------------------------
    # VALORES NÃO NULOS
    # --------------------------------------------------------

    metricas_schema.append(
        F.count(coluna_ref).alias(f"{coluna}__preenchidos")
    )

    # --------------------------------------------------------
    # VALORES NULL
    # --------------------------------------------------------

    metricas_schema.append(
        F.sum(
            F.when(coluna_ref.isNull(), 1).otherwise(0)
        ).alias(f"{coluna}__null")
    )

    # --------------------------------------------------------
    # VALORES DISTINTOS
    # --------------------------------------------------------

    metricas_schema.append(
        F.countDistinct(coluna_ref).alias(f"{coluna}__distintos")
    )

    # --------------------------------------------------------
    # VALORES VAZIOS
    # --------------------------------------------------------

    if isinstance(campo.dataType, StringType):

        metricas_schema.append(
            F.sum(
                F.when(
                    coluna_ref.isNotNull() &
                    (F.trim(coluna_ref) == ""),
                    1
                ).otherwise(0)
            ).alias(f"{coluna}__vazios")
        )

    else:

        metricas_schema.append(
            F.lit(0).cast("long").alias(f"{coluna}__vazios")
        )

print(f"Total de expressões construídas: {len(metricas_schema)}")

In [0]:
# ============================================================
# 03.6 EXECUÇÃO DO PROFILE
# ============================================================

# O QUE FAZ:
# Executa as métricas construídas anteriormente sobre o DataFrame preparado.

# COMO FAZ:
# Utiliza uma única operação de agregação para calcular, simultaneamente, as métricas estruturais de todas as colunas.

# POR QUE É IMPORTANTE:
# Reduz a quantidade de ações Spark e evita realizar uma leitura independente do DataFrame para cada coluna analisada.

# PERGUNTA RESPONDIDA:
# "Quais são os indicadores estruturais observados no DataFrame?"

schema_metrics_row = df_prepared.agg(
    *metricas_schema
).first()

print("Schema Profile executado com sucesso.")

In [0]:
# ============================================================
# 03.7 CONSTRUÇÃO DO RESULTADO
# ============================================================

# O QUE FAZ:
# Organiza as métricas agregadas em uma estrutura tabular, apresentando uma linha para cada coluna analisada.

# COMO FAZ:
# Recupera os valores das métricas calculadas na agregação anterior e transforma os resultados em registros estruturados.

# POR QUE É IMPORTANTE:
# Padroniza o resultado do Schema Profile e cria uma estrutura que poderá ser utilizada pelos demais componentes do framework, inclusive pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Quais são as características estruturais de cada coluna?"

schema_resultados = []

for campo in df_prepared.schema.fields:

    coluna = campo.name

    preenchidos = schema_metrics_row[f"{coluna}__preenchidos"]
    nulos = schema_metrics_row[f"{coluna}__null"]
    vazios = schema_metrics_row[f"{coluna}__vazios"]
    distintos = schema_metrics_row[f"{coluna}__distintos"]

    percentual_preenchimento = (
        preenchidos / total_registros * 100
        if total_registros > 0
        else 0
    )

    percentual_null = (
        nulos / total_registros * 100
        if total_registros > 0
        else 0
    )

    percentual_vazios = (
        vazios / total_registros * 100
        if total_registros > 0
        else 0
    )

    percentual_distintos = (
        distintos / total_registros * 100
        if total_registros > 0
        else 0
    )

    schema_resultados.append({
        "posicao": df_prepared.columns.index(coluna),
        "coluna": coluna,
        "tipo_spark": campo.dataType.simpleString(),
        "preenchidos": preenchidos,
        "null": nulos,
        "vazios": vazios,
        "distintos": distintos,
        "pct_preenchimento": percentual_preenchimento,
        "pct_null": percentual_null,
        "pct_vazios": percentual_vazios,
        "pct_distintos": percentual_distintos
    })

In [0]:
# ============================================================
# 03.8 DATAFRAME DO SCHEMA PROFILE
# ============================================================

# O QUE FAZ:
# Cria o DataFrame final do Schema Profile a partir das métricas calculadas para cada coluna.

# COMO FAZ:
# Converte os resultados estruturados em um DataFrame Spark e calcula a cardinalidade relativa de cada coluna.

# POR QUE É IMPORTANTE:
# Cria o principal artefato de saída do Schema Profile, disponibilizando uma estrutura padronizada para visualizações, análises posteriores e consolidação do Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é o perfil estrutural completo de cada coluna?"

schema_profile_df = (
    spark.createDataFrame(schema_resultados)
    .withColumn(
        "pct_cardinalidade",
        F.when(
            F.col("preenchidos") > 0,
            F.col("distintos") / F.col("preenchidos") * 100
        ).otherwise(F.lit(0.0))
    )
    .orderBy("posicao")
)

display(schema_profile_df)

In [0]:
# ============================================================
# 03.9 EXIBIÇÃO
# ============================================================

# O QUE FAZ:
# Exibe o resultado consolidado do Schema Profile.

# COMO FAZ:
# Utiliza a visualização nativa do Databricks sobre o DataFrame Spark produzido na etapa anterior.

# POR QUE É IMPORTANTE:
# Permite validar visualmente as métricas estruturais antes das etapas de visualização e classificação.

# PERGUNTA RESPONDIDA:
# "Como estão distribuídas as principais características estruturais das colunas?"

display(
    schema_profile_df
)

In [0]:
# ============================================================
# 03.10 VISUALIZAÇÃO — PREENCHIMENTO X NULL
# ============================================================

# O QUE FAZ:
# Prepara os indicadores de preenchimento e NULL para visualização comparativa por coluna.

# COMO FAZ:
# Seleciona somente as métricas percentuais necessárias e reorganiza os dados para facilitar a construção de um gráfico de barras no Databricks.

# POR QUE É IMPORTANTE:
# Permite identificar rapidamente colunas com baixo preenchimento ou elevada ocorrência de valores NULL.

# PERGUNTA RESPONDIDA:
# "Quais colunas apresentam problemas de preenchimento?"

visualizacao_preenchimento_null_df = (
    schema_profile_df
    .select(
        "coluna",
        "pct_preenchimento",
        "pct_null"
    )
)

display(
    visualizacao_preenchimento_null_df
)

In [0]:
# ============================================================
# 03.11 VISUALIZAÇÃO — CARDINALIDADE
# ============================================================

# O QUE FAZ:
# Prepara os indicadores de cardinalidade das colunas para visualização.

# COMO FAZ:
# Seleciona a coluna e o percentual de cardinalidade calculado pelo Schema Profile.

# POR QUE É IMPORTANTE:
# A visualização permite identificar colunas com alta ou baixa diversidade relativa de valores.

# PERGUNTA RESPONDIDA:
# "Quais colunas possuem maior ou menor diversidade de valores?"

visualizacao_cardinalidade_df = (
    schema_profile_df
    .select(
        "coluna",
        "distintos",
        "pct_cardinalidade"
    )
    .orderBy(
        F.col("pct_cardinalidade").desc()
    )
)

display(
    visualizacao_cardinalidade_df
)

In [0]:
# ============================================================
# 03.12 CLASSIFICAÇÃO ESTRUTURAL DAS COLUNAS
# ============================================================

# O QUE FAZ:
# Classifica as colunas de acordo com suas características estruturais observadas no Schema Profile.

# COMO FAZ:
# Utiliza indicadores de preenchimento e cardinalidade para identificar características como colunas completas, incompletas, de baixa cardinalidade e de alta cardinalidade.

# POR QUE É IMPORTANTE:
# Transforma métricas estatísticas em informações interpretáveis que poderão ser utilizadas posteriormente pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é a característica estrutural predominante de cada coluna?"

schema_profile_df = (
    schema_profile_df
    .withColumn(
        "classificacao_preenchimento",
        F.when(
            F.col("pct_preenchimento") == 100,
            "COMPLETA"
        )
        .when(
            F.col("pct_preenchimento") >= 95,
            "ALTA COMPLETUDE"
        )
        .when(
            F.col("pct_preenchimento") >= 80,
            "COMPLETUDE MODERADA"
        )
        .otherwise(
            "BAIXA COMPLETUDE"
        )
    )
    .withColumn(
        "classificacao_cardinalidade",
        F.when(
            F.col("pct_cardinalidade") >= 80,
            "ALTA CARDINALIDADE"
        )
        .when(
            F.col("pct_cardinalidade") >= 20,
            "CARDINALIDADE MODERADA"
        )
        .otherwise(
            "BAIXA CARDINALIDADE"
        )
    )
)

display(
    schema_profile_df
)